# Task 5 — Spark metadata stream to MongoDB

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | Spark checkpoint files, Mongo query output, and Stage 3 UI |
| Command | `bash scripts/capture_spark_evidence.sh`; `bash scripts/capture_store_evidence.sh` |
| Route | `cpg.metadata` → Spark Structured Streaming → MongoDB |
| Run date | `2026-07-23` |
```

## Approach and reasoning

Spark consumes only successful metadata events. Each micro-batch retains the
latest Kafka offset per `file_id`, then performs MongoDB replace/upsert by that
key. The persistent checkpoint resumes from committed offsets after restart.

## Key implementation excerpts

```{literalinclude} ../spark_jobs/metadata_stream_to_mongo.py
:language: python
:pyobject: write_batch
:caption: Latest-per-file MongoDB replace/upsert
```

```{literalinclude} ../spark_jobs/metadata_stream_to_mongo.py
:language: python
:lines: 137-145
:caption: Structured Streaming checkpoint and trigger
```

## Executed evidence


In [1]:
from pathlib import Path
import json

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "screenshots/stage2_manifest.json").exists()
)
manifest = json.loads((ROOT / "screenshots/stage2_manifest.json").read_text())
mongo = manifest["metrics"]["mongodb"]
spark = manifest["metrics"]["spark"]

mongo_raw = (ROOT / "screenshots/mongodb/metadata_evidence.txt").read_text().strip()
offsets_raw = (ROOT / "screenshots/spark/checkpoint_offsets.txt").read_text().strip()
commits_raw = (ROOT / "screenshots/spark/checkpoint_commits.txt").read_text().strip()

print("raw MongoDB query evidence:")
print(mongo_raw)
print("checkpoint offsets:")
print(offsets_raw)
print("checkpoint commits:")
print(commits_raw)
print("manifest summary:", {"mongodb": mongo, "spark": spark})

assert mongo["documents"] == mongo["unique_file_ids"] == mongo["unique_file_paths"] == 138
assert mongo["duplicate_file_id_groups"] == 0
assert spark["metadata_offset"] == 138
assert '"cpg.metadata":{"0":138}' in offsets_raw
assert "v1" in commits_raw


raw MongoDB query evidence:
file_metadata count: 138
unique file_id count: 138
unique file_path count: 138
repo values: ["huggingface/datasets"]
--- sample document ---
{
  _id: ObjectId('6a61bba91f1048cf3efe5cc6'),
  schema_version: '1.0',
  event_time: '2026-07-23T06:58:37.001Z',
  repo: 'huggingface/datasets',
  commit_sha: '41adfd0f9ee9ba3a6b4f719d5b551c5b19ae45e2',
  run_id: '8c3efe458b3b43eaa1399d7740848d3f',
  file_id: '16e9e0abc22af726',
  file_path: 'src/datasets/packaged_modules/text/text.py',
  file_size: Long('6862'),
  content_hash: '4f0ad7aa6bfdf82b1c7999dc7dfc9cc0a8b606b76fcdf2dbe0c29142d153b943',
  num_ast_nodes: Long('502'),
  num_cfg_edges: Long('82'),
  num_dfg_edges: Long('42'),
  num_call_edges: Long('44'),
  num_total_edges: Long('168'),
  parse_duration_ms: Long('142'),
  status: 'success'
}
--- duplicate file_id check ---
[]
checkpoint offsets:
v1
{"batchWatermarkMs":0,"batchTimestampMs":1784789932495,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.a

## What this chapter proves

| Requirement | Evidence |
|---|---|
| Spark metadata route | Embedded source subscribes to `cpg.metadata` only. |
| MongoDB write | Raw query output shows 138 documents and one unique document per file. |
| Idempotent key | Duplicate `file_id` aggregation returns an empty list. |
| Restart support | Checkpoint offset and commit files are displayed by the executed cell. |

## Final database UI evidence

```{admonition} Database UI evidence
:class: important

Mongo Express from the canonical Stage 3 replay final state on `2026-07-23`.
The UI filters the replayed `file_id`, shows its new `run_id`, and reports 138
documents after replacement.
```

![Mongo Express final document after replay](../screenshots/replay/mongodb_after_replay.png)

## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Checkpointing and replace/upsert preserved one document per file. |
| Failed | The first evidence set had query output but no MongoDB UI capture. |
| Resolution | Stage 3 added a read-only Mongo Express capture backed by the same manifest. |
```
